# Prepare 2026 Test Set

Notebook này chuẩn hóa raw test set 2026 để có schema giống `Hanoi_final.csv` trước khi đưa vào model.


In [ ]:
from pathlib import Path
import sys
import hashlib

import pandas as pd


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/test/Hanoi_test_set.csv").exists() and (PROJECT_ROOT.parent / "data/raw/test/Hanoi_test_set.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.weather_data import validate_hourly_frame, engineer_forecast_features, load_feature_selection
from src.feature_experiments import read_clean_weather

RAW_TEST_PATH = PROJECT_ROOT / "data/raw/test/Hanoi_test_set.csv"
TRAIN_FINAL_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
RAW_TRAIN_PATH = PROJECT_ROOT / "data/raw/Hanoi.csv"
FEATURE_SELECTION_PATH = PROJECT_ROOT / "data_processing/feature_selection.json"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/Hanoi_test_final.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(RAW_TEST_PATH)
print(OUTPUT_PATH)

## 1. Đọc raw test set


In [ ]:
df = read_clean_weather(RAW_TEST_PATH)
print(f"Raw test rows: {len(df):,}; columns: {len(df.columns):,}")
print(f"Time: {df['time'].min()} -> {df['time'].max()}")
df.head()

## 2. Tạo feature từ lịch sử train và test

Ghép 24 giờ cuối raw train trước khi tính lag/rolling. Chỉ dùng quan sát tại hoặc trước giờ input, giữ đủ giờ đầu của test. Feature thời gian/hướng gió dùng cùng hàm với train.

In [ ]:
load_feature_selection(FEATURE_SELECTION_PATH, "temperature_2m", 1, TRAIN_FINAL_PATH)
train_times = pd.read_csv(TRAIN_FINAL_PATH, usecols=["time"], parse_dates=["time"])["time"]
if df["time"].min() != train_times.max() + pd.Timedelta(hours=1):
    raise ValueError("External test must start exactly one hour after training data ends.")
raw_train = read_clean_weather(RAW_TRAIN_PATH)
original_test_rows = len(df)
df = engineer_forecast_features(df, history=raw_train)
assert len(df) == original_test_rows
print("Verified: causal features keep every external test hour.")

## 3. Ép schema giống `Hanoi_final.csv`


In [ ]:
train_columns = pd.read_csv(TRAIN_FINAL_PATH, nrows=0).columns.tolist()
missing_columns = [column for column in train_columns if column not in df.columns]
extra_columns = [column for column in df.columns if column not in train_columns]

print("Missing columns compared with train schema:", missing_columns)
print("Extra columns dropped:", extra_columns)

if missing_columns:
    raise ValueError(f"Test set is missing required columns: {missing_columns}")

df = df[train_columns]
print(f"Final columns: {len(df.columns):,}")
df.head()


## 4. Kiểm tra và lưu


In [ ]:
time_diff = df["time"].diff().dropna()
quality = pd.DataFrame({
    "metric": ["rows", "columns", "missing_cells", "duplicate_timestamps", "non_hourly_intervals", "start_time", "end_time"],
    "value": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df["time"].duplicated().sum()),
        int((time_diff != pd.Timedelta(hours=1)).sum()),
        df["time"].min(),
        df["time"].max(),
    ],
})
quality


In [ ]:
validate_hourly_frame(df, train_columns)

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
